# B3G — Guided SAM2 Pipeline (GPV: Guided Propose–Verify)

## Purpose

Build and evaluate a **SAM2-guided** counting pipeline in which semantic and geometric priors steer and judge SAM2, as the alternative to the anchored-correction architecture **SG** (B0 + C1 recovery + adaptive G2).

Research question for the dissertation: *how should semantic and geometric priors be combined with SAM2 for zero-shot counting?*
- **SG (anchored correction):** SAM2 runs unguided; priors patch the count.
- **GPV (guided propose–verify):** priors decide where SAM2 looks, which masks count, and how merged regions are resolved.

## Ablation ladder (each rung added once, all reported)

| Rung | Change | Prior used |
|---|---|---|
| 1 | Retain-all guided masks (End-to-End checkpoint protocol) | Semantic: where to prompt |
| 2 | **Mask deduplication** (IoU-NMS; prompt-inside-kept-mask suppression) | — |
| 3 | Multimask selection | — |
| 4 | Add B0 AMG masks to the proposal pool | — |
| 5 | Semantic verification of each mask | Semantic: what counts |
| 6 | Geometric group-mask handling (incl. placement and non-depth controls) | Geometric: how many in a merged region |

## Pre-declared parameters (fixed before any dev100 result; not tuned)

- NMS IoU threshold: **0.5**
- Score order: SAM2 predicted IoU, descending
- Group mask: area **> 4 × median** kept-mask area in the image
- Point-in-mask rule: `np.rint` + clip (as B2 / SG)

## Rules

- Development on **dev100 only**. The held-out FSC-147 cache is **not opened** until GPV and SG are both frozen (B4).
- GT is used for evaluation and diagnostics only, never at inference.
- Each rung is reported whatever its result; results decide emphasis, not whether to continue.

## Sections

| Section | Content | GPU |
|---|---|---|
| 1 | Setup: Drive, repo modules, dev100, GT, references (B0, S, SG_v1, SG_v1.1) | No |
| 2 | Rung 1 — reproduce the retain-all checkpoint counts from the cached masks | No |
| 3 | Rung 2 — deduplication + diagnostics | No |

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, versions
# ============================================================

import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"

DEV100_CSV = RESULTS_ROOT / "B1_experiments" / "subsets" / "FSC147_val_dev_100.csv"
CHECKPOINT_CACHE = (RESULTS_ROOT / "B1_experiments" / "End_to_End_Checkpoint"
                    / "guided_mask_cache")
B3_REFERENCE_FILE = (RESULTS_ROOT / "B3_B0_B2_M_Geometry_Integration" / "tables"
                     / "B3_step3F_frozen_architectures_per_image.csv")
SG_V11_PER_IMAGE = (RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "section4_SG_v1_1"
                    / "B3R_s4_SG_v1_1_per_image.csv")

B3G_DIR = RESULTS_ROOT / "B3G_Guided_SAM2_GPV"
B3G_DIR.mkdir(parents=True, exist_ok=True)

for p in [DEV100_CSV, CHECKPOINT_CACHE, B3_REFERENCE_FILE, SG_V11_PER_IMAGE]:
    assert p.exists(), f"Missing: {p}"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import importlib
import src.pipeline.io_cache as io_cache
import src.pipeline.gpv as gpv
import src.datasets as datasets_mod
for m in (io_cache, gpv, datasets_mod):
    importlib.reload(m)

import numpy, scipy, pandas
LIB_VERSIONS = {"python": sys.version.split()[0], "numpy": numpy.__version__,
                "scipy": scipy.__version__, "pandas": pandas.__version__}

print("Repo commit :", REPO_COMMIT)
for k, v in LIB_VERSIONS.items():
    print(f"{k:<7}: {v}")
print("Modules     : io_cache, gpv, datasets — loaded")
print("Output      :", B3G_DIR)

Repo commit : 73bfe97fbb7d86c1fb96b10bbb9d403260b50b71
python : 3.13.15
numpy  : 2.1.3
scipy  : 1.16.3
pandas : 2.2.3
Modules     : io_cache, gpv, datasets — loaded
Output      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3G_Guided_SAM2_GPV


In [3]:
# ============================================================
# Step 1.2 — dev100, GT points (evaluation only), references
# ============================================================

import json
import numpy as np
import pandas as pd

dev100_df = pd.read_csv(DEV100_CSV).sort_values("dataset_index").reset_index(drop=True)
assert len(dev100_df) == 100

val_ds = datasets_mod.FSC147Dataset(FSC147_RAW, split="val", load_images=False)
GT_POINTS = {}
for row in dev100_df.itertuples(index=False):
    s = val_ds[row.dataset_index]
    assert s["image_id"] == row.image_id
    (cls, cat), = s["categories"].items()
    assert cat["count"] == row.gt_count
    GT_POINTS[row.dataset_index] = np.asarray(cat["points"], dtype=np.float64).reshape(-1, 2)

ref = pd.read_csv(B3_REFERENCE_FILE)[["dataset_index", "B0_count", "S_count", "SG_count"]]
ref = ref.rename(columns={"SG_count": "SG_v1_count"})
v11 = pd.read_csv(SG_V11_PER_IMAGE)[["dataset_index", "SG_count"]].rename(
    columns={"SG_count": "SG_v1_1_count"})
reference_df = dev100_df.merge(ref, on="dataset_index").merge(v11, on="dataset_index")
assert len(reference_df) == 100

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
REFERENCE_SYSTEMS = {"B0": "B0_count", "S": "S_count",
                     "SG_v1": "SG_v1_count", "SG_v1.1": "SG_v1_1_count"}


def count_metrics(pred, gt):
    err = np.asarray(pred, float) - np.asarray(gt, float)
    nz = np.asarray(gt, float) > 0
    return {"MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
            "NAE": float((np.abs(err[nz]) / np.asarray(gt, float)[nz]).mean()),
            "mean_signed_error": float(err.mean())}


def system_summary(name, pred, frame=reference_df):
    gt = frame["gt_count"].to_numpy()
    pred = np.asarray(pred, float)
    s16 = (frame["density_stratum"] != "S6_95_100").to_numpy()
    out = {"system": name, **count_metrics(pred, gt),
           "MAE_S1_S5": float(np.abs(pred[s16] - gt[s16]).mean()),
           "MAE_S6": float(np.abs(pred[~s16] - gt[~s16]).mean())}
    for st in STRATA:
        m = (frame["density_stratum"] == st).to_numpy()
        out[f"MAE_{st[:2]}"] = float(np.abs(pred[m] - gt[m]).mean())
    return out


reference_summary_df = pd.DataFrame(
    [system_summary(n, reference_df[c]) for n, c in REFERENCE_SYSTEMS.items()])
print("Reference systems on dev100 (frozen environment):")
display(reference_summary_df.round(2))

Reference systems on dev100 (frozen environment):


,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,270.23,0.56,-77.47,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,262.24,0.43,-65.32,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1,61.93,246.59,0.44,-53.27,14.26,491.0,5.95,6.80,8.70,22.40,40.6
3,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5


# 2. Rung 1 — Retain-all guided masks (checkpoint reproduction)

The End-to-End checkpoint prompted SAM2 once per fused semantic point (`multimask_output=False`) and kept every mask: **count = number of prompts**. This section reloads those cached masks for the three checkpoint systems and verifies the cache before any change is made.

| System | Semantic source |
|---|---|
| `overall_joint` | F2: C2b regions + D6B-T-native peaks |
| `sparse_joint` | F2: C2b regions + D6B-T-native peaks (stricter) |
| `dense_recovery` | F1: C1 regions + D6B9-strict peaks |

Expected totals: 5,108 / 4,527 / 5,744 masks (15,379 in all).

In [4]:
# ============================================================
# Step 2.1 — Load and verify cached guided masks
# ============================================================

from tqdm.auto import tqdm

SYSTEMS = ["overall_joint", "sparse_joint", "dense_recovery"]
EXPECTED_TOTALS = {"overall_joint": 5108, "sparse_joint": 4527, "dense_recovery": 5744}


def guided_cache_path(system, dataset_index, image_id):
    return CHECKPOINT_CACHE / system / f"{int(dataset_index):04d}_{io_cache.image_stem(image_id)}.npz"


def load_guided(system, dataset_index, image_id):
    with np.load(guided_cache_path(system, dataset_index, image_id), allow_pickle=False) as d:
        h, w = int(d["height"]), int(d["width"])
        masks = io_cache.unpack_masks(d["packed_masks"], h, w, bitorder="little")
        out = {
            "masks": masks,
            "prompts_xy": np.asarray(d["prompt_points_xy"], dtype=np.float64).reshape(-1, 2),
            "scores": np.asarray(d["sam2_scores"], dtype=np.float64),
            "areas": np.asarray(d["mask_area_px"], dtype=np.int64),
            "prompt_inside": np.asarray(d["prompt_inside_mask"], dtype=bool),
        }
    return out


GUIDED = {s: {} for s in SYSTEMS}
cache_checks = []
for system in SYSTEMS:
    total = 0
    area_ok = inside_ok = 0
    for row in tqdm(dev100_df.itertuples(index=False), total=100, desc=system):
        g = load_guided(system, row.dataset_index, row.image_id)
        n = g["masks"].shape[0]
        assert len(g["prompts_xy"]) == len(g["scores"]) == len(g["areas"]) == n
        areas = g["masks"].reshape(n, -1).sum(axis=1) if n else np.zeros(0, int)
        area_ok += int(np.array_equal(areas, g["areas"]))
        if n:
            h, w = g["masks"].shape[1:]
            px = np.clip(np.rint(g["prompts_xy"][:, 0]).astype(int), 0, w - 1)
            py = np.clip(np.rint(g["prompts_xy"][:, 1]).astype(int), 0, h - 1)
            inside = g["masks"][np.arange(n), py, px]
            inside_ok += int(np.array_equal(inside, g["prompt_inside"]))
        else:
            inside_ok += 1
        GUIDED[system][row.dataset_index] = g
        total += n
    cache_checks.append({"system": system, "total_masks": total,
                         "expected": EXPECTED_TOTALS[system],
                         "area_match_images": area_ok,
                         "prompt_inside_match_images": inside_ok})

cache_check_df = pd.DataFrame(cache_checks)
display(cache_check_df)
CACHE_PASS = bool(((cache_check_df["total_masks"] == cache_check_df["expected"])
                   & (cache_check_df["area_match_images"] == 100)
                   & (cache_check_df["prompt_inside_match_images"] == 100)).all())
print("Guided mask cache:", "PASS" if CACHE_PASS else "FAIL")

overall_joint:   0%|          | 0/100 [00:00<?, ?it/s]

sparse_joint:   0%|          | 0/100 [00:00<?, ?it/s]

dense_recovery:   0%|          | 0/100 [00:00<?, ?it/s]

,system,total_masks,expected,area_match_images,prompt_inside_match_images
0,overall_joint,5108,5108,100,100
1,sparse_joint,4527,4527,100,100
2,dense_recovery,5744,5744,100,100


Guided mask cache: PASS


In [5]:
# ============================================================
# Step 2.2 — Rung 1 counts + mask/GT diagnostics
# ============================================================

def image_diagnostics(masks, gt_xy):
    gt_per_mask, masks_per_gt = gpv.gt_mask_incidence(masks, gt_xy)
    return {
        "masks": int(masks.shape[0]),
        "gt_covered": int((masks_per_gt > 0).sum()),
        "gt_dup_covered": int((masks_per_gt >= 2).sum()),
        "masks_no_gt": int((gt_per_mask == 0).sum()),
        "masks_gt_gt5": int((gt_per_mask > 5).sum()),
        "group_masks_4x": int(gpv.group_mask_flags(masks, 4.0).sum()),
    }


rung1_rows = []
for system in SYSTEMS:
    for row in dev100_df.itertuples(index=False):
        g = GUIDED[system][row.dataset_index]
        rung1_rows.append({"system": system, "dataset_index": row.dataset_index,
                           "density_stratum": row.density_stratum, "gt_count": row.gt_count,
                           "count": g["masks"].shape[0],
                           **image_diagnostics(g["masks"], GT_POINTS[row.dataset_index])})
rung1_df = pd.DataFrame(rung1_rows)

rung1_summary = []
for system in SYSTEMS:
    d = rung1_df[rung1_df["system"] == system].sort_values("dataset_index")
    rung1_summary.append(system_summary(f"R1 {system}", d["count"].to_numpy()))
rung1_summary_df = pd.DataFrame(rung1_summary)

print("Rung 1 (retain-all) vs references:")
display(pd.concat([reference_summary_df, rung1_summary_df]).round(2))

diag1 = (rung1_df.groupby("system")[["masks", "gt_covered", "gt_dup_covered",
                                       "masks_no_gt", "masks_gt_gt5", "group_masks_4x"]]
         .sum().reindex(SYSTEMS))
diag1["dup_share_of_covered_gt"] = diag1["gt_dup_covered"] / diag1["gt_covered"]
print("\nRung 1 diagnostics (totals over dev100):")
display(diag1.round(3))

Rung 1 (retain-all) vs references:


,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,270.23,0.56,-77.47,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,262.24,0.43,-65.32,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1,61.93,246.59,0.44,-53.27,14.26,491.0,5.95,6.80,8.70,22.40,40.6
3,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
0,R1 overall_joint,88.17,262.73,1.44,-49.37,29.97,612.0,31.80,32.80,23.00,20.75,53.0
1,R1 sparse_joint,86.42,264.01,1.26,-55.18,27.13,620.0,28.10,28.05,17.75,17.55,61.3
2,R1 dense_recovery,86.43,259.00,1.34,-43.01,30.52,589.6,21.70,34.55,37.55,24.85,37.4



Rung 1 diagnostics (totals over dev100):


,masks,gt_covered,gt_dup_covered,masks_no_gt,masks_gt_gt5,group_masks_4x,dup_share_of_covered_gt
system,,,,,,,
overall_joint,5108,6753,5407,696,341,192,0.801
sparse_joint,4527,6525,5102,629,289,156,0.782
dense_recovery,5744,8238,6534,1025,425,547,0.793


# 3. Rung 2 — Deduplication

Two pre-declared rules, applied to the cached Rung-1 masks (no new inference):

- **D1 — IoU-NMS (0.5):** visit masks by descending SAM2 score; drop a mask whose IoU with an already-kept mask exceeds 0.5.
- **D2 — D1 + prompt suppression:** after D1, visit the survivors by score; drop a mask whose **own prompt point** lies inside an already-kept mask (that object was already segmented).

Count = number of kept masks. The 0.5 threshold is fixed; Step 3.3 shows a sensitivity check only.

**What to look for:** does S1–S3 over-counting fall without losing the dense-scene recovery? How many group masks remain (input to Rung 6)?

In [6]:
# ============================================================
# Step 3.1 — Apply D1 and D2 to all systems
# ============================================================

import time

IOU_THRESHOLD = 0.5
dedup_rows, KEPT = [], {}
t0 = time.time()

for system in SYSTEMS:
    for row in tqdm(dev100_df.itertuples(index=False), total=100, desc=system):
        g = GUIDED[system][row.dataset_index]
        d1 = gpv.nms_masks(g["masks"], g["scores"], IOU_THRESHOLD)
        d2 = gpv.prompt_suppression(g["masks"], g["scores"], g["prompts_xy"], candidate_idx=d1)
        KEPT[(system, row.dataset_index)] = {"D1": d1, "D2": d2}
        for rule, idx in [("D1", d1), ("D2", d2)]:
            dedup_rows.append({"system": system, "rule": rule,
                               "dataset_index": row.dataset_index,
                               "density_stratum": row.density_stratum,
                               "gt_count": row.gt_count,
                               "count": int(len(idx)),
                               **image_diagnostics(g["masks"][idx], GT_POINTS[row.dataset_index])})

dedup_df = pd.DataFrame(dedup_rows)
print(f"Runtime: {time.time() - t0:.0f} s")

overall_joint:   0%|          | 0/100 [00:00<?, ?it/s]

sparse_joint:   0%|          | 0/100 [00:00<?, ?it/s]

dense_recovery:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 7 s


In [7]:
# ============================================================
# Step 3.2 — Rung 2 results vs Rung 1 and references
# ============================================================

rung2_summary = []
for system in SYSTEMS:
    for rule in ["D1", "D2"]:
        d = dedup_df[(dedup_df["system"] == system) & (dedup_df["rule"] == rule)].sort_values("dataset_index")
        rung2_summary.append(system_summary(f"R2-{rule} {system}", d["count"].to_numpy()))
rung2_summary_df = pd.DataFrame(rung2_summary)

comparison_df = pd.concat([reference_summary_df, rung1_summary_df, rung2_summary_df],
                          ignore_index=True)
cols = ["system", "MAE", "RMSE", "NAE", "mean_signed_error", "MAE_S1_S5", "MAE_S6",
        "MAE_S1", "MAE_S2", "MAE_S3", "MAE_S4", "MAE_S5"]
print("dev100 counting metrics (MAE_S1..S5 by density stratum):")
display(comparison_df[cols].round(2))

diag2 = (dedup_df.groupby(["system", "rule"])[["masks", "gt_covered", "gt_dup_covered",
                                               "masks_no_gt", "masks_gt_gt5", "group_masks_4x"]]
         .sum())
diag2["dup_share_of_covered_gt"] = diag2["gt_dup_covered"] / diag2["gt_covered"]
diag_all = pd.concat([diag1.assign(rule="R1").set_index("rule", append=True), diag2])
print("\nMask / GT diagnostics (totals over dev100):")
display(diag_all.round(3))

dev100 counting metrics (MAE_S1..S5 by density stratum):


,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,270.23,0.56,-77.47,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,262.24,0.43,-65.32,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1,61.93,246.59,0.44,-53.27,14.26,491.0,5.95,6.80,8.70,22.40,40.6
3,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
4,R1 overall_joint,88.17,262.73,1.44,-49.37,29.97,612.0,31.80,32.80,23.00,20.75,53.0
5,R1 sparse_joint,86.42,264.01,1.26,-55.18,27.13,620.0,28.10,28.05,17.75,17.55,61.3
6,R1 dense_recovery,86.43,259.00,1.34,-43.01,30.52,589.6,21.70,34.55,37.55,24.85,37.4
7,R2-D1 overall_joint,79.52,268.96,0.40,-77.76,17.34,639.1,2.85,3.20,7.00,24.60,80.8
8,R2-D2 overall_joint,81.62,270.41,0.41,-81.14,19.18,643.6,2.25,2.60,8.15,31.85,82.9
9,R2-D1 sparse_joint,80.91,269.73,0.42,-79.45,18.47,642.9,2.75,3.60,7.85,26.85,84.1



Mask / GT diagnostics (totals over dev100):


masks  gt_covered  gt_dup_covered  masks_no_gt  \
system         rule                                                   
overall_joint  R1     5108        6753            5407          696   
sparse_joint   R1     4527        6525            5102          629   
dense_recovery R1     5744        8238            6534         1025   
               D1     3184        8037            1117          550   
               D2     2753        7920             499          377   
overall_joint  D1     2269        6626            1200          358   
               D2     1931        6301             188          226   
sparse_joint   D1     2100        6199             622          332   
               D2     1815        6111             137          212   

                     masks_gt_gt5  group_masks_4x  dup_share_of_covered_gt  
system         rule                                                         
overall_joint  R1             341             192                    0.801  
sparse_joint   R1             289             156                    0.782  
dense_recovery R1             425             547                    0.793  
               D1              41             186                    0.139  
               D2              34             140                    0.063  
overall_joint  D1              36              99                    0.181  
               D2              27              60                    0.030  
sparse_joint   D1              29              86                    0.100  
               D2              22              54                    0.022

In [8]:
# ============================================================
# Step 3.3 — Sensitivity of D1 to the IoU threshold (diagnostic only)
# ============================================================
#
# The threshold stays fixed at 0.5. This shows how much the
# conclusion depends on that convention.
# ============================================================

sens_rows = []
for thr in [0.3, 0.5, 0.7]:
    for system in SYSTEMS:
        counts = []
        for row in dev100_df.itertuples(index=False):
            g = GUIDED[system][row.dataset_index]
            counts.append(len(gpv.nms_masks(g["masks"], g["scores"], thr)))
        s = system_summary(f"D1@{thr} {system}", counts)
        sens_rows.append({"iou_threshold": thr, "system": system,
                          "MAE": s["MAE"], "MAE_S1_S5": s["MAE_S1_S5"], "MAE_S6": s["MAE_S6"]})
sens_df = pd.DataFrame(sens_rows)
display(sens_df.pivot(index="system", columns="iou_threshold",
                      values=["MAE", "MAE_S1_S5", "MAE_S6"]).round(2))

MAE               MAE_S1_S5               MAE_S6         \
iou_threshold     0.3    0.5    0.7       0.3    0.5    0.7    0.3    0.5   
system                                                                      
dense_recovery  73.69  73.49  73.49     13.76  13.60  13.63  613.1  612.5   
overall_joint   79.70  79.52  79.33     17.41  17.34  17.24  640.3  639.1   
sparse_joint    81.14  80.91  80.67     18.60  18.47  18.31  644.0  642.9   

                       
iou_threshold     0.7  
system                 
dense_recovery  612.2  
overall_joint   638.1  
sparse_joint    641.9

In [9]:
# ============================================================
# Step 3.4 — Save Rung 1–2 artefacts
# ============================================================

from datetime import datetime, timezone

S3_DIR = B3G_DIR / "rung1_2_dedup"
S3_DIR.mkdir(parents=True, exist_ok=True)

rung1_df.to_csv(S3_DIR / "B3G_rung1_per_image.csv", index=False)
dedup_df.to_csv(S3_DIR / "B3G_rung2_per_image.csv", index=False)
comparison_df.to_csv(S3_DIR / "B3G_rung1_2_metrics.csv", index=False)
diag_all.to_csv(S3_DIR / "B3G_rung1_2_diagnostics.csv")
sens_df.to_csv(S3_DIR / "B3G_rung2_iou_sensitivity.csv", index=False)

with open(S3_DIR / "B3G_rung1_2_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3G_Guided_SAM2_GPV",
        "stage": "Rungs 1-2 (retain-all, deduplication)",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "library_versions": LIB_VERSIONS,
        "input": "End-to-End checkpoint guided mask cache (frozen environment)",
        "cache_pass": CACHE_PASS,
        "iou_threshold": IOU_THRESHOLD,
        "rules": {"D1": "IoU-NMS by descending SAM2 score",
                  "D2": "D1 + drop masks whose own prompt lies in a kept mask"},
        "new_inference": False,
        "heldout_accessed": False,
        "test_set_accessed": False,
        "gt_used_for_inference": False,
    }, f, indent=2)

print("Saved to:", S3_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3G_Guided_SAM2_GPV/rung1_2_dedup


# 4. Rungs 4, 5 and 6a (CPU, cached inputs)

## Base

`dense_recovery` + **D1** (IoU-NMS 0.5) from Rung 2 — the best Rung-2 system (MAE 73.49; S1–S5 MAE 13.60). Selected among the three existing checkpoint systems on dev100; recorded as such.

## Pre-declared rules (fixed before running; not tuned)

| Rung | Rule |
|---|---|
| **4 — B0 pool** | Pool = Rung-2 guided masks ∪ frozen B0 AMG masks; same IoU-NMS (0.5) ordered by predicted IoU (SAM2 score for guided masks, AMG `predicted_iou` for B0). Count = kept masks. |
| **5 — semantic verification** | Applied to the Rung-4 pool using the frozen SAN maps. **V1:** keep a mask if its mean target score > mean background score (SAN's own two-class decision at mask level). **V2:** keep if ≥ 50% of its pixels have target > background. |
| **6a — group multiplicity, no depth** | In the kept set, a mask with area > **4 × median** kept-mask area is a group mask and counts **max(1, round(area / median area))**; others count 1. This is the **non-depth control** for Rung 6b. |

6a is applied to R4, R5-V1 and R5-V2, so each stage's contribution can be separated.

## Inputs (all frozen dev100 caches, same environment as the checkpoint masks)

Checkpoint guided masks (Section 2), B2 B0 masks, ABC SAN maps. No new inference.

## What to look for

- **R4:** does adding B0 raise recall in S4–S6 without bringing back sparse over-counting?
- **R5:** do V1/V2 remove the no-GT masks (part-masks, background) without removing real objects in under-counted dense scenes (the B2 lesson)?
- **R6a:** how much of the S5–S6 multiplicity gap does area ratio alone close? This is the baseline Rung 6b (depth) must beat.

In [10]:
# ============================================================
# Step 4.1 — Build R4, R5 (V1, V2) and R6a for every dev100 image
# ============================================================

import importlib
importlib.reload(gpv)

assert "GUIDED" in globals() and "KEPT" in globals(), "Run Sections 2 and 3 first."

BASE_SYSTEM = "dense_recovery"
GROUP_FACTOR = 4.0

rows, mask_rows = [], []
t0 = time.time()

for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Rungs 4-6a"):
    di = row.dataset_index
    gt_xy = GT_POINTS[di]

    # ---- Rung-2 base (dense_recovery + D1)
    g = GUIDED[BASE_SYSTEM][di]
    d1 = KEPT[(BASE_SYSTEM, di)]["D1"]
    guided_masks, guided_scores = g["masks"][d1], g["scores"][d1]

    # ---- Frozen B0 masks
    b0_masks, b0_meta = io_cache.load_b0_masks(
        io_cache.cache_paths(di, row.image_id)["b0_masks"])

    # ---- Rung 4: pool + NMS
    pool, pool_scores, pool_src = gpv.pool_masks(
        [guided_masks, b0_masks],
        [guided_scores, b0_meta["predicted_iou"]],
        ["guided", "b0"],
    )
    keep4 = gpv.nms_masks(pool, pool_scores, IOU_THRESHOLD)
    r4_masks, r4_src = pool[keep4], pool_src[keep4]

    # ---- Rung 5: semantic verification on the R4 pool
    san = io_cache.load_san_scores(
        io_cache.SAN_DEV100_CACHE_DIR / io_cache.san_scores_filename(row.image_id))
    assert san["target_category"] == row.category
    ev = gpv.mask_semantic_evidence(r4_masks, san["target_score"], san["background_score"])
    v1 = gpv.verify_v1(ev)
    v2 = gpv.verify_v2(ev)

    gt_per_mask, _ = gpv.gt_mask_incidence(r4_masks, gt_xy)

    sets = {"R4": np.ones(len(r4_masks), bool), "R5-V1": v1, "R5-V2": v2}
    out = {"dataset_index": di, "image_id": row.image_id, "category": row.category,
           "density_stratum": row.density_stratum, "gt_count": row.gt_count,
           "R2_count": int(len(d1)),
           "B0_masks": int(b0_masks.shape[0]),
           "R4_from_guided": int((r4_src == "guided").sum()),
           "R4_from_b0": int((r4_src == "b0").sum())}

    for name, sel in sets.items():
        kept = r4_masks[sel]
        mult, group, med = gpv.area_ratio_multiplicity(kept, GROUP_FACTOR)
        gpm = gt_per_mask[sel]
        out[f"{name}_count"] = int(sel.sum())
        out[f"R6a({name})_count"] = int(mult.sum())
        out[f"{name}_group_masks"] = int(group.sum())
        out[f"{name}_group_pred_units"] = int(mult[group].sum())
        out[f"{name}_group_gt_inside"] = int(gpm[group].sum())

    # verification diagnostics (GT for evaluation only)
    for vname, sel in [("V1", v1), ("V2", v2)]:
        removed = ~sel
        out[f"{vname}_removed"] = int(removed.sum())
        out[f"{vname}_removed_no_gt"] = int((removed & (gt_per_mask == 0)).sum())
        out[f"{vname}_removed_with_gt"] = int((removed & (gt_per_mask > 0)).sum())
        out[f"{vname}_removed_gt_points"] = int(gt_per_mask[removed].sum())

    rows.append(out)

    for j in range(len(r4_masks)):
        mask_rows.append({"dataset_index": di, "source": r4_src[j],
                          "area": int(r4_masks[j].sum()),
                          "gt_inside": int(gt_per_mask[j]),
                          "mean_target": float(ev["mean_target"][j]),
                          "mean_background": float(ev["mean_background"][j]),
                          "target_pixel_fraction": float(ev["target_pixel_fraction"][j]),
                          "V1": bool(v1[j]), "V2": bool(v2[j])})

rung456_df = pd.DataFrame(rows).sort_values("dataset_index").reset_index(drop=True)
rung456_masks_df = pd.DataFrame(mask_rows)
print(f"Runtime: {time.time() - t0:.0f} s | R4 masks in total: {len(rung456_masks_df)}")

Rungs 4-6a:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 176 s | R4 masks in total: 3886


In [11]:
# ============================================================
# Step 4.2 — Metrics: references, R2 base, R4, R5, R6a
# ============================================================

assert (rung456_df["dataset_index"].to_numpy() == reference_df["dataset_index"].to_numpy()).all()

VARIANTS = [
    ("R2-D1 dense_recovery (base)", "R2_count"),
    ("R4 (+B0 pool)", "R4_count"),
    ("R5-V1 (+verify mean)", "R5-V1_count"),
    ("R5-V2 (+verify pixel majority)", "R5-V2_count"),
    ("R6a on R4", "R6a(R4)_count"),
    ("R6a on R5-V1", "R6a(R5-V1)_count"),
    ("R6a on R5-V2", "R6a(R5-V2)_count"),
]
rung456_summary_df = pd.DataFrame(
    [system_summary(name, rung456_df[col].to_numpy()) for name, col in VARIANTS])

table456 = pd.concat([reference_summary_df, rung456_summary_df], ignore_index=True)
print("dev100 counting metrics:")
display(table456[cols].round(2))

dev100 counting metrics:


,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,270.23,0.56,-77.47,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,262.24,0.43,-65.32,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1,61.93,246.59,0.44,-53.27,14.26,491.0,5.95,6.80,8.70,22.40,40.6
3,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
4,R2-D1 dense_recovery (base),73.49,265.07,0.39,-68.61,13.60,612.5,2.50,5.50,10.75,20.05,44.8
5,R4 (+B0 pool),69.91,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
6,R5-V1 (+verify mean),70.56,263.56,0.41,-63.20,12.82,590.2,4.05,6.65,10.35,18.30,36.7
7,R5-V2 (+verify pixel majority),70.60,263.56,0.41,-63.34,12.87,590.2,4.00,6.50,10.60,18.45,36.7
8,R6a on R4,301.48,609.47,10.42,239.86,260.77,667.9,133.15,262.10,256.35,424.50,194.7
9,R6a on R5-V1,251.45,551.94,7.20,189.45,206.33,657.5,23.75,231.15,241.95,334.05,195.2


In [12]:
# ============================================================
# Step 4.3 — Diagnostics: pool composition, verification, groups
# ============================================================

tot = rung456_df.sum(numeric_only=True)

print("RUNG 4 — POOL COMPOSITION (totals over dev100)")
print(f"  Rung-2 guided masks          : {int(tot['R2_count'])}")
print(f"  B0 AMG masks                 : {int(tot['B0_masks'])}")
print(f"  R4 kept from guided          : {int(tot['R4_from_guided'])}")
print(f"  R4 kept from B0 (added)      : {int(tot['R4_from_b0'])}")

print("\nRUNG 5 — VERIFICATION (removed from the R4 pool)")
for v in ["V1", "V2"]:
    print(f"  {v}: removed {int(tot[f'{v}_removed'])} masks "
          f"({int(tot[f'{v}_removed_no_gt'])} with no GT, "
          f"{int(tot[f'{v}_removed_with_gt'])} containing GT; "
          f"{int(tot[f'{v}_removed_gt_points'])} GT points inside removed masks)")

print("\nRUNG 6a — GROUP MASKS (area-ratio multiplicity vs GT inside)")
for s in ["R4", "R5-V1", "R5-V2"]:
    print(f"  {s:<6}: {int(tot[f'{s}_group_masks'])} group masks | "
          f"predicted units {int(tot[f'{s}_group_pred_units'])} | "
          f"GT points inside {int(tot[f'{s}_group_gt_inside'])}")

# Verification by source and by stratum
print("\nMasks removed by verification, by source:")
display(rung456_masks_df.groupby("source").agg(
    masks=("V1", "size"),
    V1_removed=("V1", lambda s: int((~s).sum())),
    V2_removed=("V2", lambda s: int((~s).sum())),
    no_gt_masks=("gt_inside", lambda s: int((s == 0).sum()))))

by_stratum_456 = rung456_df.groupby("density_stratum")[[
    "R2_count", "R4_count", "R5-V1_count", "R5-V2_count",
    "R6a(R4)_count", "R6a(R5-V1)_count", "R6a(R5-V2)_count", "gt_count"]].sum().reindex(STRATA)
print("\nTotal predicted count vs GT, by stratum:")
display(by_stratum_456)

RUNG 4 — POOL COMPOSITION (totals over dev100)
  Rung-2 guided masks          : 3184
  B0 AMG masks                 : 2298
  R4 kept from guided          : 1793
  R4 kept from B0 (added)      : 2093

RUNG 5 — VERIFICATION (removed from the R4 pool)
  V1: removed 161 masks (148 with no GT, 13 containing GT; 679 GT points inside removed masks)
  V2: removed 175 masks (160 with no GT, 15 containing GT; 716 GT points inside removed masks)

RUNG 6a — GROUP MASKS (area-ratio multiplicity vs GT inside)
  R4    : 263 group masks | predicted units 30408 | GT points inside 6266
  R5-V1 : 197 group masks | predicted units 25462 | GT points inside 5762
  R5-V2 : 187 group masks | predicted units 21208 | GT points inside 5727

Masks removed by verification, by source:


,masks,V1_removed,V2_removed,no_gt_masks
source,,,,
b0,2093,124,129,288
guided,1793,37,46,483



Total predicted count vs GT, by stratum:


,R2_count,R4_count,R5-V1_count,R5-V2_count,R6a(R4)_count,R6a(R5-V1)_count,R6a(R5-V2)_count,gt_count
density_stratum,,,,,,,,
S1_0_20,225,280,252,251,2858,646,647,195
S2_20_40,404,453,419,416,5562,4909,4425,320
S3_40_60,520,576,544,539,5638,5302,2369,537
S4_60_80,711,885,856,851,9514,7597,6755,1096
S5_80_95,763,876,864,864,2980,2953,2953,1211
S6_95_100,561,816,790,790,7479,7583,7583,6686


In [13]:
# ============================================================
# Step 4.4 — Save Rung 4–6a artefacts
# ============================================================

S4_DIR = B3G_DIR / "rung4_5_6a"
S4_DIR.mkdir(parents=True, exist_ok=True)

rung456_df.to_csv(S4_DIR / "B3G_rung4_6a_per_image.csv", index=False)
rung456_masks_df.to_csv(S4_DIR / "B3G_rung4_pool_masks.csv", index=False)
table456.to_csv(S4_DIR / "B3G_rung4_6a_metrics.csv", index=False)
by_stratum_456.to_csv(S4_DIR / "B3G_rung4_6a_totals_by_stratum.csv")

with open(S4_DIR / "B3G_rung4_6a_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3G_Guided_SAM2_GPV",
        "stage": "Rungs 4 (B0 pool), 5 (semantic verification), 6a (area-ratio groups)",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "base": f"{BASE_SYSTEM} + D1 (IoU {IOU_THRESHOLD})",
        "rules": {
            "R4": "guided D1 masks U frozen B0 masks; IoU-NMS 0.5 by predicted IoU",
            "R5-V1": "mean SAN target > mean SAN background inside mask",
            "R5-V2": ">= 50% of mask pixels with target > background",
            "R6a": f"area > {GROUP_FACTOR} x median kept area -> max(1, round(area/median))",
        },
        "inputs": "frozen dev100 caches (checkpoint guided masks, B2 B0 masks, ABC SAN maps)",
        "new_inference": False,
        "heldout_accessed": False,
        "test_set_accessed": False,
        "gt_used_for_inference": False,
    }, f, indent=2)

print("Saved to:", S4_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3G_Guided_SAM2_GPV/rung4_5_6a


# 5. Rungs 3 and 6b (GPU — SAM2 re-prompting)

## Context from Section 4

- **R4** (Rung-2 guided masks + B0 pool) is the best GPV so far: MAE 69.91; best S1–S5 of any system (12.39); S6 587.6 vs SG 491.
- **R6a** (area-ratio multiplicity on SAM2 masks) failed: group masks include background/stuff regions and the median kept area under-estimates object size. Rung 6b is designed in response, and its rules are fixed **here, before running**.

## Pre-declared rules

**Rung 3 — multimask selection.** Each `dense_recovery` prompt is re-run with `multimask_output=True`. Among the 3 masks with predicted IoU ≥ **0.8** (SAM2 AMG's default `pred_iou_thresh`) that contain the prompt, the **smallest** is kept; otherwise the top-scoring mask. Then D1 (IoU-NMS 0.5).
- **R3** = Rung 2 with multimask selection.
- **R3+R4** = R3 masks + B0 pool, IoU-NMS 0.5.

**Rung 6b — group masks re-prompted with seeds.** Applied separately to the R4 and R3+R4 kept sets.
1. **Candidates:** kept masks with area > 4 × median kept area **and** V2-verified (≥ 50% of pixels prefer target).
2. **Region:** candidate mask minus pixels covered by other non-candidate kept masks.
3. **Seeds** (one variant each), at most 200 per region, minimum spacing r = √(median kept area / π):
   - **depth** — SG's G2 split of the region (k-means k=2 on Marigold depth, 4-connected fragments, G2 retention); seeds = retained fragment centroids. *Geometric prior.*
   - **dt** — peaks of the region's distance transform. *Non-depth (shape) control.*
   - **sem** — peaks of the SAN target score inside the region. *Semantic-only control.*
4. **Re-prompt** SAM2 with each seed (single positive point, Rung-3 selection rule).
5. **Resolve:** pool = non-candidate kept masks + new masks → IoU-NMS 0.5. A candidate is **replaced** by its surviving new masks if ≥ 2 survive; otherwise it counts once.

Re-prompting is self-limiting (each seed adds at most one mask), unlike area ratio.

## Inputs and environment

Prompts: `dense_recovery` checkpoint prompts. B0 masks, SAN maps, Marigold depth: frozen dev100 caches. SAM2 masks are regenerated in the **current** environment (T4, CUDA 13.0); Step 5.3 checks how closely single-mask outputs match the cached checkpoint masks.

## Run order

T4 runtime → Sections 1–4 (CPU parts; ~5 min) → Section 5.

In [14]:
# ============================================================
# Step 5.1 — Install SAM2 (frozen commit) and build the predictor
# ============================================================
#
# Same installation as B3R Step 3.1 (repo cloned as sam2_repo, path
# added explicitly so no session restart is needed).
# ============================================================

import subprocess
import torch

assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"

import src.pipeline.b0 as b0
import src.pipeline.sg as sg
importlib.reload(b0); importlib.reload(sg); importlib.reload(gpv)

FROZEN_SAM2_COMMIT = b0.FROZEN_SAM2_COMMIT
SAM2_REPO = Path("/content/sam2_repo")
SAM2_CHECKPOINT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"

if not (SAM2_REPO / "sam2" / "__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content

sam2_commit = subprocess.check_output(
    ["git", "-C", str(SAM2_REPO), "rev-parse", "HEAD"], text=True).strip()
assert sam2_commit == FROZEN_SAM2_COMMIT, sam2_commit

if str(SAM2_REPO) not in sys.path:
    sys.path.insert(0, str(SAM2_REPO))
importlib.invalidate_caches()

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

assert b0.sha256_file(SAM2_CHECKPOINT) == b0.FROZEN_CHECKPOINT_SHA256

# Built exactly as in the End-to-End checkpoint (default build_sam2 args)
sam2_model = build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_CHECKPOINT), device=DEVICE)
sam2_model.eval()
predictor = SAM2ImagePredictor(sam2_model)

GPU_ENV = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
           "cuda": torch.version.cuda, "sam2_commit": sam2_commit}
print(GPU_ENV)
print("SAM2ImagePredictor ready")

# Free memory: only the base system's guided masks are needed now
for s in ["overall_joint", "sparse_joint"]:
    GUIDED.pop(s, None)

/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.4 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
{'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'sam2_commit': '2b90b9f5ceec907a1c18123530e92e794ad901a4'}
SAM2ImagePredictor ready


In [15]:
# ============================================================
# Step 5.2 — Rungs 3 and 6b for every dev100 image
# ============================================================

from PIL import Image

RUNG3_SCORE_THRESHOLD = 0.8
SEED_CAP = 200
SEED_TYPES = ["depth", "dt", "sem"]
SG_CFG = sg.SGConfig()           # G2 settings for depth seeds
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"


def predict_point(point_xy, multimask):
    masks, scores, _ = predictor.predict(
        point_coords=np.asarray([point_xy], dtype=np.float32),
        point_labels=np.asarray([1], dtype=np.int32),
        multimask_output=multimask,
    )
    return np.asarray(masks, dtype=bool), np.asarray(scores, dtype=np.float64)


def prompt_with_rule(points):
    out_m, out_s = [], []
    for p in points:
        m3, s3 = predict_point(p, multimask=True)
        m, s, _ = gpv.select_multimask(m3, s3, p, RUNG3_SCORE_THRESHOLD)
        out_m.append(m); out_s.append(s)
    return out_m, out_s


def r4_style_pool(guided_masks, guided_scores, b0_masks, b0_meta):
    pool, sc, src = gpv.pool_masks([guided_masks, b0_masks],
                                   [guided_scores, b0_meta["predicted_iou"]],
                                   ["guided", "b0"])
    keep = gpv.nms_masks(pool, sc, IOU_THRESHOLD)
    return pool[keep], sc[keep]


def run_6b(kept_masks, kept_scores, san, depth, seed_type):
    if kept_masks.shape[0] == 0:
        return {"count": 0, "replaced_groups": 0, "kept_groups": 0,
                "added_units": 0, "candidates": 0, "seeds": 0}
    group = gpv.group_mask_flags(kept_masks, GROUP_FACTOR)
    ev = gpv.mask_semantic_evidence(kept_masks, san["target_score"], san["background_score"])
    cand_idx = np.flatnonzero(group & gpv.verify_v2(ev))
    radius = gpv.object_radius(kept_masks)
    others = kept_masks[[i for i in range(len(kept_masks)) if i not in set(cand_idx)]]
    other_union = others.any(axis=0) if len(others) else np.zeros(kept_masks.shape[1:], bool)

    new_m, new_s, new_g, n_seeds = [], [], [], 0
    for gi in cand_idx:
        region = kept_masks[gi] & ~other_union
        if seed_type == "depth":
            seeds = gpv.seeds_depth(region, depth, SG_CFG, SEED_CAP)
        elif seed_type == "dt":
            seeds = gpv.seeds_distance_transform(region, radius, SEED_CAP)
        else:
            seeds = gpv.seeds_semantic(region, san["target_score"], radius, SEED_CAP)
        n_seeds += len(seeds)
        m, s = prompt_with_rule(seeds)
        new_m += m; new_s += s; new_g += [int(gi)] * len(m)

    res = gpv.resolve_groups(kept_masks, kept_scores, cand_idx,
                             np.asarray(new_m) if new_m else np.zeros((0,) + kept_masks.shape[1:], bool),
                             new_s, new_g, IOU_THRESHOLD)
    res.update({"candidates": int(len(cand_idx)), "seeds": int(n_seeds)})
    return res


rows5 = []
t0 = time.time()
for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Rungs 3 + 6b"):
    di = row.dataset_index
    g = GUIDED[BASE_SYSTEM][di]
    d1 = KEPT[(BASE_SYSTEM, di)]["D1"]

    image_np = np.array(Image.open(FSC147_IMAGE_DIR / row.image_id).convert("RGB"), copy=True)
    with torch.inference_mode():
        predictor.set_image(image_np)

        # ---- Rung 3: single-mask reproduction + multimask selection
        single_match, r3_m, r3_s, fallback = 0, [], [], 0
        for j, p in enumerate(g["prompts_xy"]):
            m1, _ = predict_point(p, multimask=False)
            single_match += int(np.array_equal(m1[0], g["masks"][j]))
            m3, s3 = predict_point(p, multimask=True)
            m, s, rule = gpv.select_multimask(m3, s3, p, RUNG3_SCORE_THRESHOLD)
            fallback += int(rule != "smallest_confident")
            r3_m.append(m); r3_s.append(s)

        shape = g["masks"].shape[1:]
        r3_masks = np.asarray(r3_m) if r3_m else np.zeros((0,) + shape, bool)
        r3_scores = np.asarray(r3_s, dtype=np.float64)
        r3_keep = gpv.nms_masks(r3_masks, r3_scores, IOU_THRESHOLD)

        b0_masks, b0_meta = io_cache.load_b0_masks(io_cache.cache_paths(di, row.image_id)["b0_masks"])
        san = io_cache.load_san_scores(
            io_cache.SAN_DEV100_CACHE_DIR / io_cache.san_scores_filename(row.image_id))
        depth = io_cache.load_depth(io_cache.cache_paths(di, row.image_id)["depth"])

        bases = {
            "R4": r4_style_pool(g["masks"][d1], g["scores"][d1], b0_masks, b0_meta),
            "R3R4": r4_style_pool(r3_masks[r3_keep], r3_scores[r3_keep], b0_masks, b0_meta),
        }

        out = {"dataset_index": di, "image_id": row.image_id,
               "density_stratum": row.density_stratum, "gt_count": row.gt_count,
               "prompts": len(g["prompts_xy"]), "single_mask_exact": single_match,
               "rung3_fallbacks": fallback,
               "R3_count": int(len(r3_keep)),
               "R4_count": int(len(bases["R4"][0])),
               "R3R4_count": int(len(bases["R3R4"][0])),
               "R3R4_group_masks": int(gpv.group_mask_flags(bases["R3R4"][0], GROUP_FACTOR).sum()),
               "R4_group_masks": int(gpv.group_mask_flags(bases["R4"][0], GROUP_FACTOR).sum())}

        for base_name, (km, ks) in bases.items():
            for st in SEED_TYPES:
                res = run_6b(km, ks, san, depth, st)
                for k, v in res.items():
                    out[f"6b-{st}({base_name})_{k}"] = v
    rows5.append(out)

rung36_df = pd.DataFrame(rows5).sort_values("dataset_index").reset_index(drop=True)
print(f"Runtime: {(time.time() - t0) / 60:.1f} min")

Rungs 3 + 6b:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 10.6 min


In [16]:
# ============================================================
# Step 5.3 — Environment check + Rung 3 / 6b metrics
# ============================================================

tot5 = rung36_df.sum(numeric_only=True)
print("ENVIRONMENT CHECK (single-mask SAM2 outputs vs cached checkpoint masks)")
print(f"  Prompts                 : {int(tot5['prompts'])}")
print(f"  Bit-identical masks     : {int(tot5['single_mask_exact'])} "
      f"({tot5['single_mask_exact'] / tot5['prompts']:.1%})")
print(f"  R4 recomputed == Sec. 4 : "
      f"{int((rung36_df['R4_count'].to_numpy() == rung456_df['R4_count'].to_numpy()).sum())}/100")
print(f"\nRung 3 fallbacks (no confident mask containing the prompt): "
      f"{int(tot5['rung3_fallbacks'])} of {int(tot5['prompts'])}")

VARIANTS5 = [("R2-D1 (base)", rung456_df["R2_count"]),
             ("R3 (multimask)", rung36_df["R3_count"]),
             ("R4 (+B0)", rung36_df["R4_count"]),
             ("R3+R4", rung36_df["R3R4_count"])]
for base in ["R4", "R3R4"]:
    for st in SEED_TYPES:
        VARIANTS5.append((f"6b-{st} on {base}", rung36_df[f"6b-{st}({base})_count"]))

summary5 = pd.DataFrame([system_summary(n, np.asarray(v)) for n, v in VARIANTS5])
table5 = pd.concat([reference_summary_df, summary5], ignore_index=True)
print("\ndev100 counting metrics:")
display(table5[cols].round(2))

ENVIRONMENT CHECK (single-mask SAM2 outputs vs cached checkpoint masks)
  Prompts                 : 5744
  Bit-identical masks     : 5743 (100.0%)
  R4 recomputed == Sec. 4 : 100/100

Rung 3 fallbacks (no confident mask containing the prompt): 1649 of 5744

dev100 counting metrics:


,system,MAE,RMSE,NAE,mean_signed_error,MAE_S1_S5,MAE_S6,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5
0,B0,80.69,270.23,0.56,-77.47,20.44,622.9,5.15,6.55,14.95,34.30,62.1
1,S,72.00,262.24,0.43,-65.32,14.57,588.9,4.55,5.60,8.95,24.15,44.6
2,SG_v1,61.93,246.59,0.44,-53.27,14.26,491.0,5.95,6.80,8.70,22.40,40.6
3,SG_v1.1,61.99,246.59,0.44,-53.63,14.32,491.0,5.70,6.80,8.60,22.60,41.5
4,R2-D1 (base),73.49,265.07,0.39,-68.61,13.60,612.5,2.50,5.50,10.75,20.05,44.8
5,R3 (multimask),73.50,265.08,0.39,-68.60,13.60,612.6,2.50,5.40,10.90,20.00,44.8
6,R4 (+B0),69.91,263.05,0.42,-61.59,12.39,587.6,4.65,7.05,9.35,16.95,35.5
7,R3+R4,69.94,263.06,0.42,-61.60,12.41,587.7,4.65,7.05,9.50,16.95,35.4
8,6b-depth on R4,70.23,261.84,0.47,-58.35,12.88,586.4,5.25,9.85,9.80,16.60,32.9
9,6b-dt on R4,69.94,261.42,0.46,-57.12,12.41,587.7,5.65,9.40,8.70,16.30,31.6


In [17]:
# ============================================================
# Step 5.4 — Rung 6b diagnostics
# ============================================================

diag_rows = []
for base in ["R4", "R3R4"]:
    for st in SEED_TYPES:
        p = f"6b-{st}({base})_"
        diag_rows.append({
            "variant": f"6b-{st} on {base}",
            "candidate_groups": int(tot5[p + "candidates"]),
            "seeds": int(tot5[p + "seeds"]),
            "groups_replaced": int(tot5[p + "replaced_groups"]),
            "groups_kept": int(tot5[p + "kept_groups"]),
            "units_added": int(tot5[p + "added_units"]),
        })
diag5_df = pd.DataFrame(diag_rows)
display(diag5_df)

print("\nGroup masks before 6b:  R4 =", int(tot5["R4_group_masks"]),
      "| R3+R4 =", int(tot5["R3R4_group_masks"]))

by_stratum5 = rung36_df.groupby("density_stratum")[
    ["R3_count", "R4_count", "R3R4_count"]
    + [f"6b-{st}({b})_count" for b in ["R4", "R3R4"] for st in SEED_TYPES]
    + ["gt_count"]].sum().reindex(STRATA)
print("\nTotal predicted count vs GT, by stratum:")
display(by_stratum5)

,variant,candidate_groups,seeds,groups_replaced,groups_kept,units_added
0,6b-depth on R4,190,1830,87,103,340
1,6b-dt on R4,190,2987,82,108,455
2,6b-sem on R4,190,4844,101,89,984
3,6b-depth on R3R4,192,1853,88,104,345
4,6b-dt on R3R4,192,2974,82,110,461
5,6b-sem on R3R4,192,4836,102,90,979



Group masks before 6b:  R4 = 263 | R3+R4 = 263

Total predicted count vs GT, by stratum:


,R3_count,R4_count,R3R4_count,6b-depth(R4)_count,6b-dt(R4)_count,6b-sem(R4)_count,6b-depth(R3R4)_count,6b-dt(R3R4)_count,6b-sem(R3R4)_count,gt_count
density_stratum,,,,,,,,,,
S1_0_20,225,280,280,292,300,302,291,299,304,195
S2_20_40,406,453,455,509,500,514,514,505,521,320
S3_40_60,519,576,575,611,607,660,613,607,659,537
S4_60_80,712,885,885,970,1000,1092,969,999,1093,1096
S5_80_95,763,876,875,908,947,941,905,947,934,1211
S6_95_100,560,816,815,920,979,1107,925,983,1110,6686


In [18]:
# ============================================================
# Step 5.5 — Save Rung 3 / 6b artefacts
# ============================================================

S5_DIR = B3G_DIR / "rung3_6b"
S5_DIR.mkdir(parents=True, exist_ok=True)

rung36_df.to_csv(S5_DIR / "B3G_rung3_6b_per_image.csv", index=False)
table5.to_csv(S5_DIR / "B3G_rung3_6b_metrics.csv", index=False)
diag5_df.to_csv(S5_DIR / "B3G_rung6b_diagnostics.csv", index=False)
by_stratum5.to_csv(S5_DIR / "B3G_rung3_6b_totals_by_stratum.csv")

with open(S5_DIR / "B3G_rung3_6b_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3G_Guided_SAM2_GPV",
        "stage": "Rungs 3 (multimask selection) and 6b (group re-prompting)",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "gpu_environment": GPU_ENV,
        "rules": {
            "R3": f"multimask; smallest mask with score >= {RUNG3_SCORE_THRESHOLD} containing prompt; else top score; then IoU-NMS {IOU_THRESHOLD}",
            "6b_candidates": f"area > {GROUP_FACTOR} x median kept area AND V2-verified",
            "6b_region": "candidate minus other non-candidate kept masks",
            "6b_seeds": {"depth": "SG G2 split fragment centroids",
                         "dt": "distance-transform peaks",
                         "sem": "SAN target-score peaks"},
            "6b_seed_spacing": "sqrt(median kept area / pi), min 2",
            "6b_seed_cap": SEED_CAP,
            "6b_resolve": "IoU-NMS 0.5; replace group if >= 2 new masks survive",
        },
        "single_mask_bit_identical": int(tot5["single_mask_exact"]),
        "prompts": int(tot5["prompts"]),
        "heldout_accessed": False,
        "test_set_accessed": False,
        "gt_used_for_inference": False,
    }, f, indent=2)

print("Saved to:", S5_DIR)

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3G_Guided_SAM2_GPV/rung3_6b


# 6. Dense-scene diagnostic — missing vs merged (CPU)

## Purpose

Rungs 1–6 showed **what** fails: all mask-based systems undercount dense scenes (S4–S6). This section measures **why**, before any new lever is designed.

Every GT point is classified into exactly one category, separately for three mask sets — **B0**, **R2-D1** (guided + dedup) and **R4** (best GPV):

| Category | Definition | Implied lever |
|---|---|---|
| **single** | inside at least one mask that contains only this GT point | correct |
| **merged** | inside a mask, but every covering mask contains ≥ 2 GT points | splitting / multiplicity |
| **missing — in target** | in no mask, but inside SAN's target region | more prompts |
| **missing — outside target** | in no mask, and outside SAN's target region | better semantics (resolution, tiling, D5) |

**Target region (primary):** SAN's own decision, `target_score > background_score`. A3c membership is reported alongside for reference (its fixed 10% cap labels many dense-scene objects "outside" by construction).

## Count-error decomposition

For each mask set: **count − GT ≈ surplus − merge loss − missing**, where
- **surplus** = masks containing no GT point (false positives, part-masks, background);
- **merge loss** = Σ over masks with ≥ 2 GT of (GT inside − 1);
- **missing** = GT points in no mask.

A residual is reported because overlapping masks can share GT points.

## Rules

GT is used for **diagnosis only**. No parameters are chosen here. Frozen dev100 caches only; no new inference.

## Run order

CPU runtime → Sections 1–4 → **Section 6**. Skip Section 5 (GPU only; its results are already saved).

In [19]:
# ============================================================
# Step 6.1 — Classify every GT point for B0, R2-D1 and R4
# ============================================================

assert "GUIDED" in globals() and "KEPT" in globals(), "Run Sections 1-4 first."
importlib.reload(gpv)


def classify_points(masks, gt_xy, target_region, a3c_mask):
    # Returns per-point categories and per-mask GT counts.
    n_gt = gt_xy.shape[0]
    h, w = target_region.shape
    px = np.clip(np.rint(gt_xy[:, 0]).astype(int), 0, w - 1)
    py = np.clip(np.rint(gt_xy[:, 1]).astype(int), 0, h - 1)
    in_target = target_region[py, px]
    in_a3c = a3c_mask[py, px]

    if masks.shape[0] == 0 or n_gt == 0:
        cat = np.where(in_target, "missing_in_target", "missing_outside_target")
        return cat, in_a3c, np.zeros(masks.shape[0], dtype=int)

    hits = masks[:, py, px]                  # (N masks, G points)
    gt_per_mask = hits.sum(axis=1)           # GT inside each mask
    covered = hits.any(axis=0)
    single_mask = gt_per_mask == 1
    in_single = (hits & single_mask[:, None]).any(axis=0)

    cat = np.empty(n_gt, dtype=object)
    cat[in_single] = "single"
    cat[covered & ~in_single] = "merged"
    cat[~covered & in_target] = "missing_in_target"
    cat[~covered & ~in_target] = "missing_outside_target"
    return cat, in_a3c, gt_per_mask


CATEGORIES = ["single", "merged", "missing_in_target", "missing_outside_target"]
point_rows, image_rows = [], []
t0 = time.time()

for row in tqdm(dev100_df.itertuples(index=False), total=100, desc="Diagnostic"):
    di = row.dataset_index
    gt_xy = GT_POINTS[di]

    g = GUIDED[BASE_SYSTEM][di]
    d1 = KEPT[(BASE_SYSTEM, di)]["D1"]
    b0_masks, b0_meta = io_cache.load_b0_masks(io_cache.cache_paths(di, row.image_id)["b0_masks"])

    pool, pool_scores, _ = gpv.pool_masks([g["masks"][d1], b0_masks],
                                          [g["scores"][d1], b0_meta["predicted_iou"]],
                                          ["guided", "b0"])
    r4_masks = pool[gpv.nms_masks(pool, pool_scores, IOU_THRESHOLD)]

    san = io_cache.load_san_scores(
        io_cache.SAN_DEV100_CACHE_DIR / io_cache.san_scores_filename(row.image_id))
    target_region = san["target_score"] > san["background_score"]
    a3c_mask, _ = io_cache.load_a3c_mask(io_cache.cache_paths(di, row.image_id)["a3c_masks"])

    for system, masks in [("B0", b0_masks), ("R2-D1", g["masks"][d1]), ("R4", r4_masks)]:
        cat, in_a3c, gpm = classify_points(masks, gt_xy, target_region, a3c_mask)
        counts = {c: int((cat == c).sum()) for c in CATEGORIES}
        merged_masks = gpm >= 2
        image_rows.append({
            "system": system, "dataset_index": di, "image_id": row.image_id,
            "category": row.category, "density_stratum": row.density_stratum,
            "gt_count": row.gt_count, "pred_count": int(masks.shape[0]),
            **counts,
            "missing_in_a3c": int(((cat == "missing_in_target") | (cat == "missing_outside_target"))[in_a3c].sum()) if len(cat) else 0,
            "surplus_masks": int((gpm == 0).sum()),
            "merged_masks": int(merged_masks.sum()),
            "merge_loss": int((gpm[merged_masks] - 1).sum()),
            "gt_in_merged_masks": int(gpm[merged_masks].sum()),
            "target_fraction": float(target_region.mean()),
        })

diag6_df = pd.DataFrame(image_rows)
print(f"Runtime: {time.time() - t0:.0f} s")

Diagnostic:   0%|          | 0/100 [00:00<?, ?it/s]

Runtime: 56 s


In [20]:
# ============================================================
# Step 6.2 — Missing vs merged, by stratum (% of GT points)
# ============================================================

agg = (diag6_df.groupby(["system", "density_stratum"])
       [["gt_count", "pred_count"] + CATEGORIES +
        ["missing_in_a3c", "surplus_masks", "merged_masks", "merge_loss", "gt_in_merged_masks"]]
       .sum())

pct = agg[CATEGORIES].div(agg["gt_count"], axis=0) * 100
pct.columns = [f"%{c}" for c in CATEGORIES]
pct["mean_GT_per_merged_mask"] = agg["gt_in_merged_masks"] / agg["merged_masks"].replace(0, np.nan)
pct["%missing_that_is_in_A3c"] = (agg["missing_in_a3c"]
                                  / (agg["missing_in_target"] + agg["missing_outside_target"]).replace(0, np.nan) * 100)

for system in ["B0", "R2-D1", "R4"]:
    print(f"\n=== {system}: share of GT points by category (%) ===")
    display(pct.loc[system].reindex(STRATA).round(1))


=== B0: share of GT points by category (%) ===


,%single,%merged,%missing_in_target,%missing_outside_target,mean_GT_per_merged_mask,%missing_that_is_in_A3c
density_stratum,,,,,,
S1_0_20,54.9,0.0,45.1,0.0,11.0,76.1
S2_20_40,41.9,0.0,58.1,0.0,NaN,65.6
S3_40_60,34.1,3.9,61.8,0.2,16.0,68.8
S4_60_80,34.2,24.6,41.1,0.0,40.6,55.2
S5_80_95,46.6,1.0,45.1,7.3,2.0,40.6
S6_95_100,6.4,34.6,58.4,0.7,231.3,20.8



=== R2-D1: share of GT points by category (%) ===


,%single,%merged,%missing_in_target,%missing_outside_target,mean_GT_per_merged_mask,%missing_that_is_in_A3c
density_stratum,,,,,,
S1_0_20,84.1,5.6,10.3,0.0,6.4,75.0
S2_20_40,87.2,3.8,9.1,0.0,3.9,75.9
S3_40_60,60.1,14.9,22.5,2.4,4.2,63.4
S4_60_80,47.8,39.4,12.6,0.2,11.5,37.1
S5_80_95,54.8,22.0,23.0,0.2,12.8,16.8
S6_95_100,5.8,73.2,20.8,0.2,45.4,5.8



=== R4: share of GT points by category (%) ===


,%single,%merged,%missing_in_target,%missing_outside_target,mean_GT_per_merged_mask,%missing_that_is_in_A3c
density_stratum,,,,,,
S1_0_20,89.2,2.1,8.7,0.0,6.4,82.4
S2_20_40,88.1,3.8,8.1,0.0,3.9,73.1
S3_40_60,62.8,16.8,20.3,0.2,4.3,70.0
S4_60_80,54.7,34.9,10.4,0.0,11.4,39.5
S5_80_95,62.4,20.0,17.4,0.2,12.4,21.6
S6_95_100,9.1,72.5,18.1,0.2,43.8,6.1


In [21]:
# ============================================================
# Step 6.3 — Count-error decomposition, by stratum
# ============================================================
#
#   count - GT  ~  surplus - merge_loss - missing   (+ residual)
# ============================================================

dec = agg.copy()
dec["missing_total"] = dec["missing_in_target"] + dec["missing_outside_target"]
dec["count_minus_GT"] = dec["pred_count"] - dec["gt_count"]
dec["explained"] = dec["surplus_masks"] - dec["merge_loss"] - dec["missing_total"]
dec["residual"] = dec["count_minus_GT"] - dec["explained"]

cols6 = ["gt_count", "pred_count", "count_minus_GT", "surplus_masks",
         "merge_loss", "missing_in_target", "missing_outside_target", "residual"]
for system in ["B0", "R2-D1", "R4"]:
    print(f"\n=== {system}: count-error decomposition (totals per stratum) ===")
    display(dec.loc[system, cols6].reindex(STRATA))

# Share of the undercount attributable to each cause (S4-S6, R4)
dense = dec.loc["R4"].reindex(["S4_60_80", "S5_80_95", "S6_95_100"])
deficit = dense["merge_loss"] + dense["missing_in_target"] + dense["missing_outside_target"]
share = pd.DataFrame({
    "merge_loss_%": dense["merge_loss"] / deficit * 100,
    "missing_in_target_%": dense["missing_in_target"] / deficit * 100,
    "missing_outside_target_%": dense["missing_outside_target"] / deficit * 100,
})
print("\nR4 — share of the dense-scene deficit by cause (%):")
display(share.round(1))


=== B0: count-error decomposition (totals per stratum) ===


,gt_count,pred_count,count_minus_GT,surplus_masks,merge_loss,missing_in_target,missing_outside_target,residual
density_stratum,,,,,,,,
S1_0_20,195,184,-11,62,10,88,0,25
S2_20_40,320,209,-111,61,0,186,0,14
S3_40_60,537,266,-271,61,60,332,1,61
S4_60_80,1096,514,-582,63,317,451,0,123
S5_80_95,1211,636,-575,26,6,546,89,40
S6_95_100,6686,489,-6197,36,2303,3902,46,18



=== R2-D1: count-error decomposition (totals per stratum) ===


,gt_count,pred_count,count_minus_GT,surplus_masks,merge_loss,missing_in_target,missing_outside_target,residual
density_stratum,,,,,,,,
S1_0_20,195,225,30,51,38,20,0,37
S2_20_40,320,404,84,115,26,29,0,24
S3_40_60,537,520,-17,155,121,121,13,83
S4_60_80,1096,711,-385,121,569,138,2,203
S5_80_95,1211,763,-448,53,533,278,2,312
S6_95_100,6686,561,-6125,55,5245,1389,16,470



=== R4: count-error decomposition (totals per stratum) ===


,gt_count,pred_count,count_minus_GT,surplus_masks,merge_loss,missing_in_target,missing_outside_target,residual
density_stratum,,,,,,,,
S1_0_20,195,280,85,95,38,17,0,45
S2_20_40,320,453,133,161,26,26,0,24
S3_40_60,537,576,39,195,132,109,1,86
S4_60_80,1096,885,-211,168,571,114,0,306
S5_80_95,1211,876,-335,68,535,211,2,345
S6_95_100,6686,816,-5870,84,5312,1213,16,587



R4 — share of the dense-scene deficit by cause (%):


,merge_loss_%,missing_in_target_%,missing_outside_target_%
density_stratum,,,
S4_60_80,83.4,16.6,0.0
S5_80_95,71.5,28.2,0.3
S6_95_100,81.2,18.5,0.2


In [22]:
# ============================================================
# Step 6.4 — Per-image view of S4-S6 (R4) and save
# ============================================================

dense_imgs = diag6_df[(diag6_df["system"] == "R4")
                      & diag6_df["density_stratum"].isin(["S4_60_80", "S5_80_95", "S6_95_100"])].copy()
dense_imgs["%merged"] = dense_imgs["merged"] / dense_imgs["gt_count"] * 100
dense_imgs["%missing_in_target"] = dense_imgs["missing_in_target"] / dense_imgs["gt_count"] * 100
dense_imgs["%missing_outside"] = dense_imgs["missing_outside_target"] / dense_imgs["gt_count"] * 100
print("R4, S5-S6 images (sorted by GT):")
display(dense_imgs[dense_imgs["density_stratum"] != "S4_60_80"]
        .sort_values("gt_count", ascending=False)[[
            "image_id", "category", "density_stratum", "gt_count", "pred_count",
            "%merged", "%missing_in_target", "%missing_outside", "target_fraction"]].round(1))

S6_DIR = B3G_DIR / "section6_dense_diagnostic"
S6_DIR.mkdir(parents=True, exist_ok=True)
diag6_df.to_csv(S6_DIR / "B3G_s6_per_image_categories.csv", index=False)
pct.to_csv(S6_DIR / "B3G_s6_category_shares_by_stratum.csv")
dec.to_csv(S6_DIR / "B3G_s6_error_decomposition_by_stratum.csv")

with open(S6_DIR / "B3G_s6_metadata.json", "w") as f:
    json.dump({
        "notebook": "B3G_Guided_SAM2_GPV",
        "stage": "Section 6 — dense-scene diagnostic (missing vs merged)",
        "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "repo_commit": REPO_COMMIT,
        "mask_sets": ["B0 (frozen)", f"R2-D1 ({BASE_SYSTEM})", "R4 (R2-D1 + B0, IoU-NMS 0.5)"],
        "target_region": "SAN target_score > background_score (A3c reported for reference)",
        "categories": CATEGORIES,
        "gt_use": "diagnosis only",
        "new_inference": False,
        "heldout_accessed": False,
        "test_set_accessed": False,
    }, f, indent=2)
print("\nSaved to:", S6_DIR)

R4, S5-S6 images (sorted by GT):


,image_id,category,density_stratum,gt_count,pred_count,%merged,%missing_in_target,%missing_outside,target_fraction
89,935.jpg,birds,S6_95_100,2092,12,99.0,0.7,0.0,1.0
281,6969.jpg,birds,S6_95_100,949,26,97.2,2.0,0.1,0.6
170,3665.jpg,toilet paper rolls,S6_95_100,907,14,97.4,0.6,1.7,0.4
98,975.jpg,birds,S6_95_100,718,71,70.6,23.1,0.0,1.0
77,840.jpg,books,S6_95_100,637,56,15.1,80.4,0.0,1.0
278,6931.jpg,birds,S6_95_100,355,97,30.4,53.2,0.0,1.0
293,7436.jpg,pills,S6_95_100,283,286,0.7,0.4,0.0,1.0
233,5866.jpg,grapes,S6_95_100,267,117,10.1,58.8,0.0,1.0
2,215.jpg,grapes,S6_95_100,259,120,6.2,57.5,0.0,1.0
140,3426.jpg,polka dots,S6_95_100,219,17,99.1,0.0,0.0,0.0



Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3G_Guided_SAM2_GPV/section6_dense_diagnostic


# B3G conclusion

**Rungs (dev100, MAE / S1–S5 / S6):**
- R1 retain-all (dense_recovery): 86.43 / — / — — checkpoint reproduced.
- **R2 dedup (D1, IoU-NMS 0.5):** sparse-scene failure was duplicate masks, not semantic points (S1 21.70 → 2.50). D2 (prompt suppression) rejected; IoU threshold insensitive (0.3–0.7).
- R3 multimask selection: no effect (merging is not a part/whole ambiguity).
- **R4 (+B0 pool): best GPV — 69.91 / 12.39 / 587.6** (S1–S5 best of all systems; SG_v1: 61.93 / 14.26 / 491.0).
- R5 semantic verification: 92% precise but slightly worse (system undercounts).
- R6a area-ratio multiplicity: fails (background group masks; poor size reference).
- R6b group re-prompting: small, mixed; depth seeds ≈ distance-transform seeds; semantic seeds trade S1–S5 for S6.

**Section 6 diagnostic:** dense-scene (S4–S6) deficit is ~75–80% **merging** and ~20% **missing**; guided prompting converts B0's "missing" into "merged". SAN's target-vs-"background" region covers almost the whole image in dense scenes (weak negative prompt).

**Continued in:** `B3GK_GPV_Multiplicity_and_Prompts.ipynb` — merged-mask rule, multiplicity options K1–K5, prompting P1, and the GPV freeze.